In [ ]:
""""
PSIms Complete Unified System v3.7 - MODERN UI + EXACT STABLE ENGINE
Pharma Stakeholder Interaction Monitoring System

Author: Subhoit Talukdar
Version: 3.7 - Production Ready (Tabs + Bootstrap)

UI: Modern 'ttkbootstrap' with TABS (Data Prep / Analysis).
LOGIC: Exact v3.1 Stable Engine (No optimizations).
STABILITY: Config crash protection included.
"""

# Check if running in Jupyter and handle kernel restart
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None and 'IPKernelApp' in ipython.config:
        IS_JUPYTER = True
        import atexit
        def cleanup_on_exit():
            try:
                import gc
                gc.collect()
            except:
                pass
        atexit.register(cleanup_on_exit)
    else:
        IS_JUPYTER = False
except:
    IS_JUPYTER = False

import pandas as pd
import numpy as np
import os
import json
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext
from pathlib import Path
from datetime import datetime
import openpyxl
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from difflib import SequenceMatcher

# UI Library Check
try:
    import ttkbootstrap as ttk
    from ttkbootstrap.constants import *
    from ttkbootstrap.scrolled import ScrolledText
    THEME_AVAILABLE = True
except ImportError:
    import tkinter.ttk as ttk
    from tkinter.scrolledtext import ScrolledText
    THEME_AVAILABLE = False
    print("NOTE: ttkbootstrap not found. UI will look standard.")

# Optional visualization libraries
try:
    import matplotlib
    matplotlib.use('Agg')  # Non-interactive backend
    import matplotlib.pyplot as plt
    import seaborn as sns
    VISUALIZATION_AVAILABLE = True
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("Note: matplotlib/seaborn not installed. Visualizations will be disabled.")

import warnings
warnings.filterwarnings('ignore')


# =====================================================
# CONFIGURATION MANAGER (SAFE MODE)
# =====================================================

class PSImsConfig:
    """Manages system configuration with persistent storage and crash protection"""
    
    def __init__(self):
        self.config_file = 'psims_config.json.backup'
        self.config = self.load_or_create()
    
    def load_or_create(self):
        """Load config or create with default structure"""
        default_config = {
            'folders': {
                'input_folder': '',
                'csv_output': '',
                'results_output': ''
            },
            'settings': {
                'eligibility_mode': 'relaxed',
                'num_clusters': 6,
                'require_engagement': True,
                'zero_data_position': 'first',
                'naming_method': 'combined',
                'high_threshold': 30,
                'low_threshold': 15,
                'show_quality_metrics': True,
                'generate_visualizations': False
            },
            'last_run': None
        }
        
        if os.path.exists(self.config_file):
            try:
                with open(self.config_file, 'r') as f:
                    content = f.read()
                    if not content.strip(): 
                        raise ValueError("Empty config file")
                    
                    loaded_config = json.loads(content)P
                    
                    # Ensure all required keys exist
                    for key in default_config:
                        if key not in loaded_config:
                            loaded_config[key] = default_config[key]
                        elif isinstance(default_config[key], dict):
                            for subkey in default_config[key]:
                                if subkey not in loaded_config[key]:
                                    loaded_config[key][subkey] = default_config[key][subkey]
                    
                    return loaded_config
            except Exception as e:
                # --- CRITICAL FIX: Delete corrupt file to prevent crash loops ---
                print(f"Warning: Config file corrupted ({e}). Resetting to defaults.")
                try:
                    os.remove(self.config_file)
                except:
                    pass
                return default_config
        
        return default_config
    
    def save(self):
        """Save configuration to file"""
        try:
            with open(self.config_file, 'w') as f:
                json.dump(self.config, f, indent=4)
        except Exception as e:
            print(f"Warning: Could not save config: {e}")
    
    def update_folder(self, key, path):
        """Update folder path in config"""
        if 'folders' not in self.config:
            self.config['folders'] = {}
        self.config['folders'][key] = path
        self.save()
    
    def get_folder(self, key):
        """Get folder path from config"""
        if 'folders' not in self.config:
            self.config['folders'] = {}
        return self.config['folders'].get(key, '')
    
    def update_setting(self, key, value):
        """Update a setting"""
        if 'settings' not in self.config:
            self.config['settings'] = {}
        self.config['settings'][key] = value
        self.save()
    
    def get_setting(self, key, default=None):
        """Get a setting"""
        if 'settings' not in self.config:
            self.config['settings'] = {}
        return self.config['settings'].get(key, default)


# =====================================================
# SCORING CONFIGURATION (EXACT COPY)
# =====================================================

SCORING_CONFIG = {
    'engagement': {
        'email_open_weight': 0.50,
        'email_click_weight': 0.50,
        'whatsapp_read_weight': 0.50,
        'whatsapp_click_weight': 0.50,
        'call_productive_weight': 0.70,
        'call_duration_weight': 0.30,
        'final_email_weight': 0.33,
        'final_whatsapp_weight': 0.33,
        'final_call_weight': 0.34,
        'max_score': 100
    },
    'academic': {
        'publication_points_per_item': 10,
        'publication_max_score': 50,
        'trial_points_per_item': 20,
        'trial_max_score': 30,
        'association_points_per_item': 10,
        'association_max_score': 20,
        'max_score': 100,
        'baseline_threshold': 10
    },
    'social_media': {
        'follower_log_multiplier': 10,
        'follower_max_score': 50,
        'follower_min_threshold': 100,
        'platform_points_per_item': 10,
        'platform_max_score': 30,
        'healthcare_platform_points_per_item': 10,
        'healthcare_platform_max_score': 20,
        'max_score': 100
    },
    'recognition': {
        'award_points_per_item': 15,
        'award_max_score': 30,
        'press_points_per_item': 10,
        'press_max_score': 25,
        'event_points_per_item': 8,
        'event_max_score': 25,
        'association_points_per_item': 5,
        'association_max_score': 20,
        'max_score': 100
    }
}


# =====================================================
# SMART EXCEL CONVERTER (EXACT COPY)
# =====================================================

class SmartConverter:
    """Intelligently combines and converts Excel files"""
    
    def __init__(self, output_folder, log_callback=None):
        self.output_folder = output_folder
        self.log = log_callback or print
        self.warnings = []
    
    def standardize_name(self, name):
        """Standardize names (lowercase, no spaces)"""
        return name.strip().lower().replace(' ', '_').replace("'", "")
    
    def fuzzy_match(self, str1, str2, threshold=0.8):
        """Fuzzy string matching"""
        return SequenceMatcher(None, str1.lower(), str2.lower()).ratio() >= threshold
    
    def combine_pi_batches(self, pi_files):
        """Combine multiple PI batch files"""
        self.log("\n📊 Combining Personal Info Batches...")
        
        all_sheets_per_file = {}
        for file in pi_files:
            try:
                wb = openpyxl.load_workbook(file, read_only=True)
                all_sheets_per_file[file] = wb.sheetnames
                wb.close()
                self.log(f"   {os.path.basename(file)}: {len(all_sheets_per_file[file])} sheets")
            except Exception as e:
                self.log(f"   ❌ Error loading {os.path.basename(file)}: {e}")
                continue
        
        if not all_sheets_per_file:
            self.log("   ❌ No valid PI files to process")
            return
        
        master_sheets = all_sheets_per_file[list(all_sheets_per_file.keys())[0]]
        sheet_mapping = {sheet: [] for sheet in master_sheets}
        
        for file in all_sheets_per_file.keys():
            file_sheets = all_sheets_per_file[file]
            for master_sheet in master_sheets:
                if master_sheet in file_sheets:
                    sheet_mapping[master_sheet].append((file, master_sheet))
                else:
                    for file_sheet in file_sheets:
                        if self.fuzzy_match(master_sheet, file_sheet):
                            warning = f"Fuzzy match: '{master_sheet}' → '{file_sheet}' in {os.path.basename(file)}"
                            self.warnings.append(warning)
                            self.log(f"   ⚠️  {warning}")
                            sheet_mapping[master_sheet].append((file, file_sheet))
                            break
        
        COUNT_BASED_SHEETS = ['publication', 'trials', 'academic_association', 
                             'awards', 'press', 'events', 'association']
        
        self.log("\n   📝 Combining sheets...")
        for sheet_name, file_sheet_pairs in sheet_mapping.items():
            if not file_sheet_pairs:
                self.log(f"   ⚠️  No data found for sheet: {sheet_name}")
                continue
            
            dfs = []
            for file, actual_sheet_name in file_sheet_pairs:
                try:
                    df = pd.read_excel(file, sheet_name=actual_sheet_name)
                    df.columns = [self.standardize_name(col) for col in df.columns]
                    dfs.append(df)
                    self.log(f"      ✓ {os.path.basename(file)} → {actual_sheet_name}: {len(df)} rows")
                except Exception as e:
                    self.log(f"      ❌ Error reading {actual_sheet_name} from {os.path.basename(file)}: {e}")
            
            if dfs:
                combined = pd.concat(dfs, ignore_index=True)
                standardized_sheet = self.standardize_name(sheet_name)
                
                if 'uin' in combined.columns:
                    is_count_based = any(cb_sheet in standardized_sheet for cb_sheet in COUNT_BASED_SHEETS)
                    
                    if not is_count_based and standardized_sheet in ['pi', 'clinics', 'digital', 'healthcare_platforms']:
                        before = len(combined)
                        combined = combined.drop_duplicates(subset=['uin'], keep='last')
                        after = len(combined)
                        if before != after:
                            self.log(f"          Removed {before - after} duplicate UINs (reference sheet)")
                    else:
                        self.log(f"          Kept all {len(combined)} rows (count-based sheet)")
                
                output_name = standardized_sheet + '.csv'
                output_path = os.path.join(self.output_folder, output_name)
                combined.to_csv(output_path, index=False, encoding='utf-8')
                self.log(f"      ✓ Saved {output_name}: {len(combined)} rows, {len(combined.columns)} cols")
    
    def convert_engagement_files(self, engagement_files):
        """Convert engagement files"""
        self.log("\n📅 Converting Engagement Files...")
        
        for file in engagement_files:
            try:
                df = pd.read_excel(file)
                df.columns = [self.standardize_name(col) for col in df.columns]
                
                original_name = os.path.basename(file).replace('.xlsx', '').replace('.xls', '')
                standardized_name = self.standardize_name(original_name) + '.csv'
                
                output_path = os.path.join(self.output_folder, standardized_name)
                df.to_csv(output_path, index=False, encoding='utf-8')
                
                self.log(f"   ✓ {original_name} → {standardized_name}: {len(df)} rows")
            except Exception as e:
                self.log(f"   ❌ Failed to convert {os.path.basename(file)}: {e}")
    
    def get_warnings(self):
        return self.warnings


# =====================================================
# PSIMS ANALYSIS ENGINE (EXACT COPY + FILTER)
# =====================================================

class PSImsEngine:
    """Core analytics engine with all enhancements and fixes"""
    
    def __init__(self, csv_folder, output_folder, log_callback, 
             eligibility_mode, require_engagement,
             naming_method, high_threshold, low_threshold,
             show_quality_metrics, generate_visualizations,
             zero_data_position, selected_csv_files, 
             selected_specialty='All Specialties'):
        
        self.csv_folder = csv_folder
        self.output_folder = output_folder
        self.log = log_callback or print
        self.data = {}
        
        # Dynamic parameters
        self.eligibility_mode = eligibility_mode
        self.require_engagement = require_engagement
        self.naming_method = naming_method
        self.high_threshold = high_threshold
        self.low_threshold = low_threshold
        self.show_quality_metrics = show_quality_metrics
        self.enable_visualizations = generate_visualizations
        self.zero_data_position = zero_data_position
        self.selected_csv_files = selected_csv_files or []
        self.selected_specialty = selected_specialty
        
        # Eligibility rules
        self.eligibility_rules = {
            'strict': {'min_buckets': 4},
            'moderate': {'min_buckets': 3},
            'relaxed': {'min_buckets': 2},
            'basic': {'min_buckets': 1},
            'custom': {'min_buckets': 2}
        }
        
        self.log(f"🔧 Engine initialized:")
        self.log(f"   Mode: {self.eligibility_mode}")
        self.log(f"   Specialty Filter: {self.selected_specialty}")
        self.log(f"   Thresholds: High={self.high_threshold}, Low={self.low_threshold}")

    def load_csv(self, filename):
        """Safely load CSV with encoding fallback"""
        filepath = os.path.join(self.csv_folder, filename)
        try:
            if os.path.exists(filepath):
                df = pd.read_csv(filepath, encoding='utf-8', low_memory=False)
                return df
            return pd.DataFrame()
        except:
            try:
                return pd.read_csv(filepath, encoding='latin-1', low_memory=False)
            except:
                return pd.DataFrame()
    
    def should_load_file(self, filename):
        """Check if file should be loaded based on selection"""
        if not self.selected_csv_files or len(self.selected_csv_files) == 0:
            return True
        return filename in self.selected_csv_files
    
    def standardize_uin_column(self, df, is_engagement=False):
        """Standardize UIN column name to lowercase 'uin'"""
        if df.empty:
            return df
        
        for col in df.columns:
            if col.lower() == 'uin':
                if col != 'uin':
                    df = df.rename(columns={col: 'uin'})
                return df
        
        uin_variations = [
            'Client doctor code', 'client doctor code', 'CLIENT DOCTOR CODE',
            'Client_doctor_code', 'client_doctor_code', 'Client_Doctor_Code',
            'Doctor Code', 'doctor code', 'DoctorCode', 'doctorcode',
            'doctor_id', 'DoctorID', 'Doctor_ID',
            'Client Code', 'client code', 'ClientCode'
        ]
        
        for col in df.columns:
            if col in uin_variations:
                df = df.rename(columns={col: 'uin'})
                return df
        
        for col in df.columns:
            col_lower = col.lower().replace('_', '').replace(' ', '')
            if any(term in col_lower for term in ['doctorcode', 'clientcode', 'doctorid']):
                df = df.rename(columns={col: 'uin'})
                return df
        
        return df
    
    def standardize_engagement_columns(self, df):
        """Standardize engagement metric column names"""
        column_mappings = {
            'HCP Email Open Rate': 'hcp_email_open_rate',
            'HCP Email Click Rate': 'hcp_email_click_rate',
            'HCP Whatsapp Read Rate': 'hcp_whatsapp_read_rate',
            'HCP Whatsapp Click Rate': 'hcp_whatsapp_click_rate',
            'HCP Call Productive Rate': 'hcp_call_productive_rate',
            'Average Duration Connected Calls': 'average_duration_connected_calls'
        }
        
        rename_dict = {}
        for col in df.columns:
            for original, standardized in column_mappings.items():
                if col.lower() == original.lower():
                    rename_dict[col] = standardized
                    break
        
        if rename_dict:
            df = df.rename(columns=rename_dict)
        
        return df
    
    def load_all_data(self):
        """Load all CSV files"""
        self.log("\n📂 Loading Data Files...")
        
        files = {
            'pi': 'pi.csv',
            'clinics': 'clinics.csv',
            'publications': 'publication.csv',
            'trials': 'trials.csv',
            'academic_associations': 'academic_association.csv',
            'digital': 'digital.csv',
            'healthcare_platforms': 'healthcare_platforms.csv',
            'awards': 'awards.csv',
            'press': 'press.csv',
            'events': 'events.csv',
            'associations': 'association.csv'
        }
        
        for key, filename in files.items():
            if self.should_load_file(filename):
                self.data[key] = self.load_csv(filename)
                if not self.data[key].empty:
                    self.data[key] = self.standardize_uin_column(self.data[key])
                    if 'uin' in self.data[key].columns:
                        self.log(f"   ✓ {key}: {len(self.data[key])} rows")
                    else:
                        self.log(f"   ⚠️ {key}: No UIN column")
                        self.data[key] = pd.DataFrame()
                else:
                    self.data[key] = pd.DataFrame()
            else:
                self.data[key] = pd.DataFrame()
                self.log(f"   ⊘ {key}: Skipped (not selected)")
        
        engagement_files = [f for f in os.listdir(self.csv_folder) 
                          if f.endswith('.csv') and self.should_load_file(f) and
                          any(month in f.lower() for month in ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 
                                                               'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])]
        
        engagement_dfs = []
        for eng_file in engagement_files:
            df = self.load_csv(eng_file)
            if not df.empty:
                self.log(f"   📋 {eng_file}: {len(df)} rows")
                df = self.standardize_uin_column(df, is_engagement=True)
                df = self.standardize_engagement_columns(df)
                
                if 'uin' in df.columns:
                    engagement_dfs.append(df)
                    self.log(f"      ✓ {df['uin'].nunique()} unique UINs found")
        
        if engagement_dfs:
            combined = pd.concat(engagement_dfs, ignore_index=True)
            
            expected_cols = {
                'hcp_email_open_rate': 0,
                'hcp_email_click_rate': 0,
                'hcp_whatsapp_read_rate': 0,
                'hcp_whatsapp_click_rate': 0,
                'hcp_call_productive_rate': 0,
                'average_duration_connected_calls': 0
            }
            
            for col, default in expected_cols.items():
                if col not in combined.columns:
                    combined[col] = default
                else:
                    combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(default)
            
            agg_dict = {col: 'mean' for col in expected_cols.keys()}
            self.data['engagement'] = combined.groupby('uin', as_index=False).agg(agg_dict)
            self.log(f"   ✓ Engagement (averaged): {len(self.data['engagement'])} UINs")
        else:
            self.data['engagement'] = pd.DataFrame()

    def aggregate_by_uin(self):
        """Aggregate all data sources by UIN - Now includes Specialty Filtering"""
        self.log("\n🔗 Aggregating by UIN...")
        
        if self.data['pi'].empty:
            self.log("   ❌ No PI data")
            return pd.DataFrame()
        
        # --- NEW LOGIC: Pull uin_specialty and Filter ---
        pi_df = self.data['pi'].copy()
        
        # Ensure we have the specialty column
        if 'uin_specialty' not in pi_df.columns:
            if 'specialty' in pi_df.columns:
                 pi_df.rename(columns={'specialty': 'uin_specialty'}, inplace=True)
            else:
                 pi_df['uin_specialty'] = 'Unknown'

        # Apply Specialty Filter BEFORE processing
        if self.selected_specialty and self.selected_specialty != "All Specialties":
            original_count = len(pi_df)
            pi_df = pi_df[pi_df['uin_specialty'].astype(str).str.lower() == self.selected_specialty.lower()]
            filtered_count = len(pi_df)
            self.log(f"   🔎 Filtered by Specialty '{self.selected_specialty}': {original_count} → {filtered_count} doctors")
            
            if pi_df.empty:
                self.log("   ❌ No doctors found for this specialty.")
                return pd.DataFrame()
        # ------------------------------------------------
        
        master = pi_df[['uin']].drop_duplicates().copy()
        self.log(f"   Master: {len(master)} UINs")
        
        for metric in ['publication_count', 'trial_count', 'academic_association_count',
                      'award_count', 'platform_count', 'total_followers',
                      'healthcare_platform_count', 'press_count', 'event_count', 
                      'association_count']:
            master[metric] = 0
            
        # Publications
        if not self.data['publications'].empty and 'uin' in self.data['publications'].columns:
            pubs_df = self.data['publications'].copy()
            if 'publication_type' in pubs_df.columns:
                pubs_df = pubs_df[pubs_df['publication_type'].notna() & (pubs_df['publication_type'] != '')]
            
            if len(pubs_df) > 0:
                pubs = pubs_df.groupby('uin').size().reset_index(name='publication_count')
                master = master.merge(pubs, on='uin', how='left', suffixes=('', '_new'))
                master['publication_count'] = master['publication_count_new'].fillna(0)
                master.drop(['publication_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Publications: {(master['publication_count'] > 0).sum()} doctors, {master['publication_count'].sum():.0f} total")
        
        # Trials
        if not self.data['trials'].empty and 'uin' in self.data['trials'].columns:
            trials_df = self.data['trials'].copy()
            trial_col = None
            for col in ['trial_type', 'trial_id', 'trial_title']:
                if col in trials_df.columns:
                    trial_col = col
                    break
            
            if trial_col:
                trials_df = trials_df[trials_df[trial_col].notna() & (trials_df[trial_col] != '')]
            
            if len(trials_df) > 0:
                trials = trials_df.groupby('uin').size().reset_index(name='trial_count')
                master = master.merge(trials, on='uin', how='left', suffixes=('', '_new'))
                master['trial_count'] = master['trial_count_new'].fillna(0)
                master.drop(['trial_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Trials: {(master['trial_count'] > 0).sum()} doctors")
        
        # Academic Associations
        if not self.data['academic_associations'].empty and 'uin' in self.data['academic_associations'].columns:
            acad_df = self.data['academic_associations'].copy()
            acad_col = None
            for col in ['association_type', 'association_name', 'association_title']:
                if col in acad_df.columns:
                    acad_col = col
                    break
            
            if acad_col:
                acad_df = acad_df[acad_df[acad_col].notna() & (acad_df[acad_col] != '')]
            
            if len(acad_df) > 0:
                acad = acad_df.groupby('uin').size().reset_index(name='academic_association_count')
                master = master.merge(acad, on='uin', how='left', suffixes=('', '_new'))
                master['academic_association_count'] = master['academic_association_count_new'].fillna(0)
                master.drop(['academic_association_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Academic Assoc: {(master['academic_association_count'] > 0).sum()} doctors")
        
        # Awards
        if not self.data['awards'].empty and 'uin' in self.data['awards'].columns:
            awards_df = self.data['awards'].copy()
            if 'award_level' in awards_df.columns:
                awards_df = awards_df[awards_df['award_level'].notna() & (awards_df['award_level'] != '')]
            
            if len(awards_df) > 0:
                awards = awards_df.groupby('uin').size().reset_index(name='award_count')
                master = master.merge(awards, on='uin', how='left', suffixes=('', '_new'))
                master['award_count'] = master['award_count_new'].fillna(0)
                master.drop(['award_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Awards: {(master['award_count'] > 0).sum()} doctors")
        
        # Press
        if not self.data['press'].empty and 'uin' in self.data['press'].columns:
            press_df = self.data['press'].copy()
            if 'press_type' in press_df.columns:
                press_df = press_df[press_df['press_type'].notna() & (press_df['press_type'] != '')]
            
            if len(press_df) > 0:
                press = press_df.groupby('uin').size().reset_index(name='press_count')
                master = master.merge(press, on='uin', how='left', suffixes=('', '_new'))
                master['press_count'] = master['press_count_new'].fillna(0)
                master.drop(['press_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Press: {(master['press_count'] > 0).sum()} doctors")
        
        # Events
        if not self.data['events'].empty and 'uin' in self.data['events'].columns:
            events_df = self.data['events'].copy()
            if 'event_type' in events_df.columns:
                events_df = events_df[events_df['event_type'].notna() & (events_df['event_type'] != '')]
            
            if len(events_df) > 0:
                events = events_df.groupby('uin').size().reset_index(name='event_count')
                master = master.merge(events, on='uin', how='left', suffixes=('', '_new'))
                master['event_count'] = master['event_count_new'].fillna(0)
                master.drop(['event_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Events: {(master['event_count'] > 0).sum()} doctors")
        
        # Associations
        if not self.data['associations'].empty and 'uin' in self.data['associations'].columns:
            assoc_df = self.data['associations'].copy()
            if 'association_type' in assoc_df.columns:
                assoc_df = assoc_df[assoc_df['association_type'].notna() & (assoc_df['association_type'] != '')]
            
            if len(assoc_df) > 0:
                assoc = assoc_df.groupby('uin').size().reset_index(name='association_count')
                master = master.merge(assoc, on='uin', how='left', suffixes=('', '_new'))
                master['association_count'] = master['association_count_new'].fillna(0)
                master.drop(['association_count_new'], axis=1, errors='ignore', inplace=True)
                self.log(f"   ✓ Associations: {(master['association_count'] > 0).sum()} doctors")
        
        # Digital data
        if not self.data['digital'].empty and 'uin' in self.data['digital'].columns:
            digital_clean = self.data['digital'][(self.data['digital']['uin'].notna()) & 
                                                 (self.data['digital']['uin'] != '')].copy()
            if len(digital_clean) > 0:
                if 'sm_channel' in digital_clean.columns:
                    digital_clean = digital_clean[digital_clean['sm_channel'].notna() & (digital_clean['sm_channel'] != '')]
                    agg_dict = {'platform_count': ('sm_channel', 'nunique')}
                    if 'sm_followers' in digital_clean.columns:
                        digital_clean['sm_followers'] = pd.to_numeric(digital_clean['sm_followers'], errors='coerce').fillna(0)
                        agg_dict['total_followers'] = ('sm_followers', 'sum')
                    digital = digital_clean.groupby('uin').agg(**agg_dict).reset_index()
                else:
                    digital = digital_clean.groupby('uin').size().reset_index(name='platform_count')
                    digital['total_followers'] = 0
                
                master = master.merge(digital, on='uin', how='left', suffixes=('_old', ''))
                master.drop([c for c in master.columns if c.endswith('_old')], axis=1, errors='ignore', inplace=True)
                master['platform_count'] = master.get('platform_count', 0).fillna(0)
                master['total_followers'] = master.get('total_followers', 0).fillna(0)
                self.log(f"   ✓ Social Media: {(master['platform_count'] > 0).sum()} doctors")
        
        # Healthcare platforms
        if not self.data['healthcare_platforms'].empty and 'uin' in self.data['healthcare_platforms'].columns:
            hc = self.data['healthcare_platforms'].groupby('uin').size().reset_index(name='healthcare_platform_count')
            master = master.merge(hc, on='uin', how='left', suffixes=('', '_new'))
            master['healthcare_platform_count'] = master['healthcare_platform_count_new'].fillna(0)
            master.drop(['healthcare_platform_count_new'], axis=1, errors='ignore', inplace=True)
        
        # Merge engagement
        if not self.data['engagement'].empty:
            merged = master.merge(self.data['engagement'], on='uin', how='left')
            for col in ['hcp_email_open_rate', 'hcp_email_click_rate', 'hcp_whatsapp_read_rate',
                       'hcp_whatsapp_click_rate', 'hcp_call_productive_rate', 
                       'average_duration_connected_calls']:
                if col in merged.columns:
                    merged[col] = merged[col].fillna(0)
        else:
            merged = master.copy()
            for col in ['hcp_email_open_rate', 'hcp_email_click_rate', 'hcp_whatsapp_read_rate',
                       'hcp_whatsapp_click_rate', 'hcp_call_productive_rate', 
                       'average_duration_connected_calls']:
                merged[col] = 0
        
        # Merge PI details - UPDATED TO INCLUDE uin_specialty
        pi_cols = ['uin'] + [c for c in ['full_name', 'mobile', 'whatsapp_phone', 
                                     'email_id_1', 'uin_specialty'] if c in pi_df.columns]
        
        merged = merged.merge(pi_df[pi_cols].drop_duplicates('uin'), on='uin', how='left')
        
        # Rename uin_specialty to specialty for output consistency
        if 'uin_specialty' in merged.columns:
            merged.rename(columns={'uin_specialty': 'specialty'}, inplace=True)

        # Merge clinic data
        if not self.data['clinics'].empty and 'uin' in self.data['clinics'].columns:
            clinic_cols = ['uin'] + [c for c in ['clinic_address', 'clinic_city', 'clinic_state'] 
                                   if c in self.data['clinics'].columns]
            if len(clinic_cols) > 1:
                merged = merged.merge(self.data['clinics'][clinic_cols].drop_duplicates('uin'), on='uin', how='left')
                self.log(f"   ✓ Merged clinic data (including city)")
        
        merged.fillna(0, inplace=True)
        return merged
    
    def calculate_scores(self, df):
        """Calculate 4 bucket scores"""
        self.log("\n🎯 Calculating Scores...")
        
        def calc_engagement(row):
            config = SCORING_CONFIG['engagement']
            
            email_open = row.get('hcp_email_open_rate', 0) or 0
            email_click = row.get('hcp_email_click_rate', 0) or 0
            wa_read = row.get('hcp_whatsapp_read_rate', 0) or 0
            wa_click = row.get('hcp_whatsapp_click_rate', 0) or 0
            call_prod = row.get('hcp_call_productive_rate', 0) or 0
            call_dur = row.get('average_duration_connected_calls', 0) or 0
            
            if 0 < email_open <= 1:
                email_open = email_open * 100
            if 0 < email_click <= 1:
                email_click = email_click * 100
            if 0 < wa_read <= 1:
                wa_read = wa_read * 100
            if 0 < wa_click <= 1:
                wa_click = wa_click * 100
            if 0 < call_prod <= 1:
                call_prod = call_prod * 100
            
            email_score = (email_open * config['email_open_weight'] + 
                          email_click * config['email_click_weight'])
            wa_score = (wa_read * config['whatsapp_read_weight'] + 
                        wa_click * config['whatsapp_click_weight'])
            
            call_dur_norm = min((call_dur / 60) * 100, 100) if call_dur > 0 else 0
            call_score = (call_prod * config['call_productive_weight'] + 
                          call_dur_norm * config['call_duration_weight'])
            
            final = (email_score * config['final_email_weight'] +
                    wa_score * config['final_whatsapp_weight'] +
                    call_score * config['final_call_weight'])
            
            return round(final, 2)
        
        def calc_academic(row):
            config = SCORING_CONFIG['academic']
            pubs = row.get('publication_count', 0) or 0
            trials = row.get('trial_count', 0) or 0
            acad_assoc = row.get('academic_association_count', 0) or 0
            
            pub_score = min(pubs * config['publication_points_per_item'], 
                           config['publication_max_score'])
            trial_score = min(trials * config['trial_points_per_item'], 
                             config['trial_max_score'])
            assoc_score = min(acad_assoc * config['association_points_per_item'], 
                             config['association_max_score'])
            
            total = pub_score + trial_score + assoc_score
            return round(min(total, config['max_score']), 2)
        
        def calc_social(row):
            config = SCORING_CONFIG['social_media']
            platforms = row.get('platform_count', 0) or 0
            followers = row.get('total_followers', 0) or 0
            hc_platforms = row.get('healthcare_platform_count', 0) or 0
            
            if followers >= config['follower_min_threshold']:
                follower_score = min(np.log10(followers) * config['follower_log_multiplier'],
                                   config['follower_max_score'])
            else:
                follower_score = 0
            
            platform_score = min(platforms * config['platform_points_per_item'],
                               config['platform_max_score'])
            hc_score = min(hc_platforms * config['healthcare_platform_points_per_item'],
                          config['healthcare_platform_max_score'])
            
            total = follower_score + platform_score + hc_score
            return round(min(total, config['max_score']), 2)
        
        def calc_recognition(row):
            config = SCORING_CONFIG['recognition']
            awards = row.get('award_count', 0) or 0
            press = row.get('press_count', 0) or 0
            events = row.get('event_count', 0) or 0
            assoc = row.get('association_count', 0) or 0
            
            award_score = min(awards * config['award_points_per_item'],
                            config['award_max_score'])
            press_score = min(press * config['press_points_per_item'],
                            config['press_max_score'])
            event_score = min(events * config['event_points_per_item'],
                            config['event_max_score'])
            assoc_score = min(assoc * config['association_points_per_item'],
                            config['association_max_score'])
            
            total = award_score + press_score + event_score + assoc_score
            return round(min(total, config['max_score']), 2)
        
        df['engagement_score'] = df.apply(calc_engagement, axis=1)
        df['academic_score'] = df.apply(calc_academic, axis=1)
        df['social_media_score'] = df.apply(calc_social, axis=1)
        df['recognition_score'] = df.apply(calc_recognition, axis=1)
        
        df['engagement_percentile'] = df['engagement_score'].rank(pct=True)
        df['academic_percentile'] = df['academic_score'].rank(pct=True)
        df['social_media_percentile'] = df['social_media_score'].rank(pct=True)
        df['recognition_percentile'] = df['recognition_score'].rank(pct=True)
        
        df['engagement_data_available'] = df['engagement_score'] > 0
        df['academic_data_available'] = (df['publication_count'] + df['trial_count'] + df['academic_association_count']) > 0
        df['social_media_data_available'] = (df['platform_count'] + df['healthcare_platform_count'] + df['total_followers']) > 0
        df['recognition_data_available'] = (df['award_count'] + df['press_count'] + df['event_count'] + df['association_count']) > 0
        
        df['buckets_with_data'] = (df['engagement_data_available'].astype(int) +
                                   df['academic_data_available'].astype(int) +
                                   df['social_media_data_available'].astype(int) +
                                   df['recognition_data_available'].astype(int))
        
        rules = self.eligibility_rules.get(self.eligibility_mode, self.eligibility_rules['relaxed'])
        df['eligible_for_clustering'] = (
            (df['buckets_with_data'] >= rules['min_buckets']) &
            (df['engagement_data_available'] if self.require_engagement else True)
        )
        
        def get_reason(row):
            if row['eligible_for_clustering']:
                return ''
            reasons = []
            if self.require_engagement and not row['engagement_data_available']:
                reasons.append('No engagement data')
            if row['buckets_with_data'] < rules['min_buckets']:
                reasons.append(f"Only {int(row['buckets_with_data'])}/{rules['min_buckets']} buckets")
            return '; '.join(reasons)
        
        df['insufficient_data_reason'] = df.apply(get_reason, axis=1)
        
        self.log(f"   ✓ Scored {len(df)} doctors")
        self.log(f"   Avg Engagement: {df['engagement_score'].mean():.2f}")
        self.log(f"   Avg Academic: {df['academic_score'].mean():.2f}")
        self.log(f"   Avg Social Media: {df['social_media_score'].mean():.2f}")
        self.log(f"   Avg Recognition: {df['recognition_score'].mean():.2f}")
        
        return df
    
    def get_smart_cluster_name(self, cluster_data):
        """Generate smart cluster name using ACTUAL scores and percentiles"""
        
        avg_scores = {
            'Engagement': cluster_data['engagement_score'].mean(),
            'Academic': cluster_data['academic_score'].mean(),
            'Social Media': cluster_data['social_media_score'].mean(),
            'Recognition': cluster_data['recognition_score'].mean()
        }
        
        avg_percentiles = {
            'Engagement': cluster_data['engagement_percentile'].mean(),
            'Academic': cluster_data['academic_percentile'].mean(),
            'Social Media': cluster_data['social_media_percentile'].mean(),
            'Recognition': cluster_data['recognition_percentile'].mean()
        }
        
        max_score_bucket = max(avg_scores, key=avg_scores.get)
        max_score = avg_scores[max_score_bucket]
        max_percentile = avg_percentiles[max_score_bucket]
        
        all_scores = list(avg_scores.values())
        
        if all(score < self.low_threshold for score in all_scores):
            return "Low Activity Profile"
        
        buckets_with_data = sum(1 for score in all_scores if score > 5)
        if buckets_with_data == 1:
            return f"Single-Bucket: {max_score_bucket}"
        
        score_range = max(all_scores) - min(all_scores)
        if score_range < 10 and max_score < 25:
            return "Balanced Low Profile"
        elif score_range < 10:
            return "Balanced Profile"
        
        if buckets_with_data <= 3 and max_score < 20:
            return f"Emerging: {max_score_bucket}"
        
        if self.naming_method == 'absolute':
            if max_score > self.high_threshold:
                return f"{max_score_bucket}-Focused"
            elif max_score > 20:
                return f"{max_score_bucket}-Leaning"
            else:
                return f"Low {max_score_bucket}"
        
        elif self.naming_method == 'percentile':
            if max_percentile > 0.90:
                return f"High {max_score_bucket}"
            elif max_percentile > 0.75:
                return f"Above Average {max_score_bucket}"
            elif max_percentile > 0.50:
                return f"Moderate {max_score_bucket}"
            else:
                return f"Low {max_score_bucket}"
        
        else:  # combined (default)
            if max_score > self.high_threshold and max_percentile > 0.75:
                other_scores = [s for b, s in avg_scores.items() if b != max_score_bucket]
                if other_scores and max_score > 1.5 * max(other_scores):
                    return f"{max_score_bucket}-Focused"
                else:
                    return f"{max_score_bucket}-Leaning"
            
            elif max_score > 20 or max_percentile > 0.50:
                return f"Moderate {max_score_bucket}"
            
            else:
                return f"Low {max_score_bucket}"
    
    def perform_clustering(self, df, n_clusters=6):
        """Cluster with proper naming using ACTUAL scores"""
        self.log(f"\n🔍 Clustering (k={n_clusters}, mode={self.eligibility_mode})...")
        
        zero_data = df[df['buckets_with_data'] == 0].copy()
        has_data = df[df['buckets_with_data'] > 0].copy()
        
        self.log(f"   Zero-data doctors: {len(zero_data)}")
        self.log(f"   Doctors with data: {len(has_data)}")
        
        cluster_profiles = []
        silhouette_scores = {}
        
        if len(zero_data) > 0:
            if self.zero_data_position == 'exclude':
                self.log("   ⊘ Zero-data doctors excluded from output")
            else:
                if self.zero_data_position == 'first':
                    zero_data['cluster_id'] = 1
                    zero_data['cluster_name'] = 'Cluster 1: No Data'
                elif self.zero_data_position == 'last':
                    zero_data['cluster_id'] = n_clusters
                    zero_data['cluster_name'] = f'Cluster {n_clusters}: No Data'
                else:
                    zero_data['cluster_id'] = 0
                    zero_data['cluster_name'] = 'Cluster 0: No Data'
                
                cluster_profiles.append({
                    'cluster_id': zero_data['cluster_id'].iloc[0],
                    'count': len(zero_data),
                    'avg_engagement': 0,
                    'avg_academic': 0,
                    'avg_social_media': 0,
                    'avg_recognition': 0,
                    'cluster_name': zero_data['cluster_name'].iloc[0]
                })
        
        if len(has_data) >= (n_clusters - 1):
            score_cols = ['engagement_score', 'academic_score', 'social_media_score', 'recognition_score']
            for col in score_cols:
                has_data[col] = has_data[col].fillna(0)
            
            features = has_data[score_cols].values
            features = np.nan_to_num(features, nan=0.0)
            
            scaler = StandardScaler()
            features_scaled = scaler.fit_transform(features)
            
            if self.zero_data_position == 'exclude' or len(zero_data) == 0:
                kmeans_clusters = n_clusters
                cluster_id_offset = 1
            else:
                kmeans_clusters = n_clusters - 1
                cluster_id_offset = 2 if self.zero_data_position == 'first' else 1
            
            kmeans = KMeans(n_clusters=kmeans_clusters, random_state=42, n_init=10)
            clusters = kmeans.fit_predict(features_scaled)
            has_data['cluster_id'] = clusters + cluster_id_offset
            
            if self.show_quality_metrics and len(has_data) > kmeans_clusters:
                try:
                    silhouette_avg = silhouette_score(features_scaled, clusters)
                    silhouette_scores['overall'] = silhouette_avg
                    self.log(f"   📊 Silhouette Score: {silhouette_avg:.3f}")
                except:
                    pass
            
            for i in range(kmeans_clusters):
                cluster_data = has_data[has_data['cluster_id'] == i + cluster_id_offset]
                
                cluster_name = self.get_smart_cluster_name(cluster_data)
                full_cluster_name = f"Cluster {i + cluster_id_offset}: {cluster_name}"
                
                has_data.loc[has_data['cluster_id'] == i + cluster_id_offset, 'cluster_name'] = full_cluster_name
                
                profile = {
                    'cluster_id': i + cluster_id_offset,
                    'count': len(cluster_data),
                    'avg_engagement': cluster_data['engagement_score'].mean(),
                    'avg_academic': cluster_data['academic_score'].mean(),
                    'avg_social_media': cluster_data['social_media_score'].mean(),
                    'avg_recognition': cluster_data['recognition_score'].mean(),
                    'cluster_name': full_cluster_name
                }
                cluster_profiles.append(profile)
                self.log(f"   {full_cluster_name}: {len(cluster_data)} doctors")
        else:
            has_data['cluster_id'] = 1 if self.zero_data_position != 'first' else 2
            has_data['cluster_name'] = f"Cluster {has_data['cluster_id'].iloc[0]}: Mixed Profile"
            cluster_profiles.append({
                'cluster_id': has_data['cluster_id'].iloc[0],
                'count': len(has_data),
                'avg_engagement': has_data['engagement_score'].mean(),
                'avg_academic': has_data['academic_score'].mean(),
                'avg_social_media': has_data['social_media_score'].mean(),
                'avg_recognition': has_data['recognition_score'].mean(),
                'cluster_name': has_data['cluster_name'].iloc[0]
            })
        
        if self.zero_data_position == 'exclude':
            result = has_data
        else:
            result = pd.concat([zero_data, has_data], ignore_index=True) if len(zero_data) > 0 else has_data
        
        return result, cluster_profiles, silhouette_scores
    
    def create_visualizations(self, df, cluster_profiles):
        """Generate cluster visualizations"""
        if not self.enable_visualizations:
            return None
        
        if not VISUALIZATION_AVAILABLE:
            self.log("\n⚠️  Visualization libraries not installed (matplotlib/seaborn)")
            return None
        
        self.log("\n📈 Generating Visualizations...")
        
        try:
            fig, axes = plt.subplots(2, 2, figsize=(16, 12))
            fig.suptitle('PSIMS Cluster Analysis', fontsize=16, fontweight='bold')
            
            cluster_counts = df['cluster_name'].value_counts()
            axes[0, 0].barh(range(len(cluster_counts)), cluster_counts.values)
            axes[0, 0].set_yticks(range(len(cluster_counts)))
            axes[0, 0].set_yticklabels([name.split(':')[1].strip() if ':' in name else name 
                                     for name in cluster_counts.index], fontsize=9)
            axes[0, 0].set_xlabel('Number of Physicians')
            axes[0, 0].set_title('Cluster Size Distribution')
            axes[0, 0].grid(axis='x', alpha=0.3)
            
            profiles_df = pd.DataFrame(cluster_profiles)
            score_cols = ['avg_engagement', 'avg_academic', 'avg_social_media', 'avg_recognition']
            x = np.arange(len(profiles_df))
            width = 0.2
            
            for idx, col in enumerate(score_cols):
                offset = (idx - 1.5) * width
                axes[0, 1].bar(x + offset, profiles_df[col], width, 
                              label=col.replace('avg_', '').replace('_', ' ').title())
            
            axes[0, 1].set_xlabel('Cluster ID')
            axes[0, 1].set_ylabel('Average Score')
            axes[0, 1].set_title('Average Scores by Cluster')
            axes[0, 1].set_xticks(x)
            axes[0, 1].set_xticklabels(profiles_df['cluster_id'])
            axes[0, 1].legend()
            axes[0, 1].grid(axis='y', alpha=0.3)
            
            has_data = df[df['buckets_with_data'] > 0]
            score_data = [has_data['engagement_score'], has_data['academic_score'],
                         has_data['social_media_score'], has_data['recognition_score']]
            axes[1, 0].boxplot(score_data, labels=['Engagement', 'Academic', 'Social Media', 'Recognition'])
            axes[1, 0].set_ylabel('Score')
            axes[1, 0].set_title('Score Distribution Across All Physicians')
            axes[1, 0].grid(axis='y', alpha=0.3)
            
            bucket_counts = df['buckets_with_data'].value_counts().sort_index()
            axes[1, 1].bar(bucket_counts.index, bucket_counts.values, color='steelblue')
            axes[1, 1].set_xlabel('Number of Buckets with Data')
            axes[1, 1].set_ylabel('Number of Physicians')
            axes[1, 1].set_title('Data Availability Distribution')
            axes[1, 1].grid(axis='y', alpha=0.3)
            
            plt.tight_layout()
            
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            viz_file = os.path.join(self.output_folder, f'cluster_visualization_{timestamp}.png')
            plt.savefig(viz_file, dpi=300, bbox_inches='tight')
            plt.close()
            
            self.log(f"   ✓ Saved: {viz_file}")
            return viz_file
            
        except Exception as e:
            self.log(f"   ⚠️ Visualization generation failed: {e}")
            return None
    
    def generate_output(self, df, cluster_profiles, silhouette_scores):
        """Save comprehensive results"""
        self.log("\n💾 Generating Output...")
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = os.path.join(self.output_folder, f'psims_results_{timestamp}.csv')
        
        output_cols = ['uin', 'full_name', 'specialty', 'mobile', 'clinic_city',
                      'engagement_score', 'engagement_percentile', 'engagement_data_available',
                      'academic_score', 'academic_percentile', 'academic_data_available',
                      'social_media_score', 'social_media_percentile', 'social_media_data_available',
                      'recognition_score', 'recognition_percentile', 'recognition_data_available',
                      'buckets_with_data', 'eligible_for_clustering',
                      'cluster_id', 'cluster_name', 'insufficient_data_reason']
        
        output_cols = [c for c in output_cols if c in df.columns]
        output_df = df[output_cols]
        
        output_df.to_csv(output_file, index=False, encoding='utf-8')
        self.log(f"   ✓ Saved: {output_file}")
        
        profiles_file = None
        if cluster_profiles:
            profiles_df = pd.DataFrame(cluster_profiles)
            profiles_file = os.path.join(self.output_folder, f'cluster_profiles_{timestamp}.csv')
            profiles_df.to_csv(profiles_file, index=False, encoding='utf-8')
            self.log(f"   ✓ Saved: {profiles_file}")
            
            self.log("\n📊 Cluster Summary:")
            for _, profile in profiles_df.iterrows():
                pct = (profile['count'] / len(df)) * 100
                self.log(f"   • {profile['cluster_name']}: {profile['count']} ({pct:.1f}%)")
                self.log(f"      Eng: {profile['avg_engagement']:.2f} | Acad: {profile['avg_academic']:.2f} | "
                        f"Social: {profile['avg_social_media']:.2f} | Recog: {profile['avg_recognition']:.2f}")
        
        summary_file = os.path.join(self.output_folder, f'summary_stats_{timestamp}.txt')
        with open(summary_file, 'w', encoding='utf-8') as f:
            f.write("=" * 80 + "\n")
            f.write("PSIMS ANALYSIS SUMMARY REPORT\n")
            f.write("=" * 80 + "\n\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Eligibility Mode: {self.eligibility_mode.upper()}\n")
            f.write(f"Require Engagement: {self.require_engagement}\n")
            f.write(f"Naming Method: {self.naming_method}\n")
            f.write(f"Thresholds: High={self.high_threshold}, Low={self.low_threshold}\n\n")
            
            f.write("-" * 80 + "\n")
            f.write("OVERALL STATISTICS\n")
            f.write("-" * 80 + "\n")
            f.write(f"Total Physicians: {len(df)}\n")
            f.write(f"Eligible for Clustering: {df['eligible_for_clustering'].sum()}\n")
            f.write(f"Insufficient Data: {(~df['eligible_for_clustering']).sum()}\n\n")
            
            f.write("-" * 80 + "\n")
            f.write("DATA AVAILABILITY\n")
            f.write("-" * 80 + "\n")
            for bucket in ['engagement', 'academic', 'social_media', 'recognition']:
                col = f'{bucket}_data_available'
                if col in df.columns:
                    count = df[col].sum()
                    pct = (count / len(df)) * 100
                    f.write(f"{bucket.replace('_', ' ').title()}: {count} ({pct:.1f}%)\n")
            f.write("\n")
            
            f.write("-" * 80 + "\n")
            f.write("SCORE STATISTICS\n")
            f.write("-" * 80 + "\n")
            for bucket in ['engagement', 'academic', 'social_media', 'recognition']:
                score_col = f'{bucket}_score'
                if score_col in df.columns:
                    f.write(f"\n{bucket.replace('_', ' ').title()}:\n")
                    f.write(f"  Mean: {df[score_col].mean():.2f}\n")
                    f.write(f"  Median: {df[score_col].median():.2f}\n")
                    f.write(f"  Std Dev: {df[score_col].std():.2f}\n")
                    f.write(f"  Min: {df[score_col].min():.2f}\n")
                    f.write(f"  Max: {df[score_col].max():.2f}\n")
            
            if cluster_profiles:
                f.write("\n" + "-" * 80 + "\n")
                f.write("CLUSTER DETAILS\n")
                f.write("-" * 80 + "\n")
                for profile in cluster_profiles:
                    pct = (profile['count'] / len(df)) * 100
                    f.write(f"\n{profile['cluster_name']}\n")
                    f.write(f"  Size: {profile['count']} physicians ({pct:.1f}%)\n")
                    f.write(f"  Avg Engagement: {profile['avg_engagement']:.2f}\n")
                    f.write(f"  Avg Academic: {profile['avg_academic']:.2f}\n")
                    f.write(f"  Avg Social Media: {profile['avg_social_media']:.2f}\n")
                    f.write(f"  Avg Recognition: {profile['avg_recognition']:.2f}\n")
            
            if silhouette_scores and self.show_quality_metrics:
                f.write("\n" + "-" * 80 + "\n")
                f.write("CLUSTER QUALITY METRICS\n")
                f.write("-" * 80 + "\n")
                if 'overall' in silhouette_scores:
                    score = silhouette_scores['overall']
                    f.write(f"Silhouette Score: {score:.3f}\n")
                    if score > 0.7:
                        f.write("Quality: Excellent - Strong cluster separation\n")
                    elif score > 0.5:
                        f.write("Quality: Good - Reasonable cluster structure\n")
                    elif score > 0.3:
                        f.write("Quality: Fair - Weak cluster structure\n")
                    else:
                        f.write("Quality: Poor - Consider reducing cluster count\n")
            
            f.write("\n" + "=" * 80 + "\n")
        
        self.log(f"   ✓ Saved: {summary_file}")
        
        # Generate visualizations if enabled
        viz_file = self.create_visualizations(df, cluster_profiles)
        
        return output_file, profiles_file, summary_file, viz_file
    
    def run_analysis(self, n_clusters=6):
        """Execute complete analysis pipeline"""
        self.log("\n" + "="*60)
        self.log("🚀 STARTING PSIMS ANALYSIS")
        self.log("="*60)
        
        self.load_all_data()
        aggregated = self.aggregate_by_uin()
        
        if aggregated.empty:
            self.log("\n❌ No data to analyze")
            return None, None, None, None
        
        scored = self.calculate_scores(aggregated)
        clustered, profiles, silhouette = self.perform_clustering(scored, n_clusters)
        
        output_file, profiles_file, summary_file, viz_file = self.generate_output(clustered, profiles, silhouette)
        
        self.log("\n" + "="*60)
        self.log("✅ ANALYSIS COMPLETE!")
        self.log("="*60)
        
        return output_file, profiles_file, summary_file, viz_file


# =====================================================
# GUI APPLICATION (REWRITTEN FOR TABS + BOOTSTRAP)
# =====================================================

class PSImsGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("PSIMS v3.7 - Modern UI")
        self.root.geometry("1100x850")
        
        self.config = PSImsConfig()
        
        # Initialize Variables (same as v3.1 but cleaned up)
        self.input_folder_var = tk.StringVar(value=self.config.get_folder('input_folder'))
        self.csv_folder_var = tk.StringVar(value=self.config.get_folder('csv_output'))
        self.results_folder_var = tk.StringVar(value=self.config.get_folder('results_output'))
        
        self.eligibility_var = tk.StringVar(value=self.config.get_setting('eligibility_mode', 'relaxed'))
        self.clusters_var = tk.StringVar(value=str(self.config.get_setting('num_clusters', 6)))
        self.require_eng_var = tk.BooleanVar(value=self.config.get_setting('require_engagement', True))
        self.zero_pos_var = tk.StringVar(value=self.config.get_setting('zero_data_position', 'first'))
        self.naming_var = tk.StringVar(value=self.config.get_setting('naming_method', 'combined'))
        self.high_thresh_var = tk.StringVar(value=str(self.config.get_setting('high_threshold', 30)))
        self.low_thresh_var = tk.StringVar(value=str(self.config.get_setting('low_threshold', 15)))
        
        self.specialty_var = tk.StringVar(value='All Specialties')
        self.show_metrics_var = tk.BooleanVar(value=self.config.get_setting('show_quality_metrics', True))
        self.generate_viz_var = tk.BooleanVar(value=self.config.get_setting('generate_visualizations', False))
        
        self.selected_pi_files = []
        self.selected_eng_files = []
        self.selected_csv_files = []
        
        # Build UI
        self.create_widgets()

    def create_widgets(self):
        # --- HEADER ---
        if THEME_AVAILABLE:
            header_frame = ttk.Frame(self.root, bootstyle="secondary")
            header_frame.pack(fill='x', ipady=10)
            ttk.Label(header_frame, text=" PSIMS v3.7 - Production Ready ", 
                     font=('Helvetica', 16, 'bold'), bootstyle="inverse-secondary").pack(side=LEFT, padx=10)
        else:
            tk.Label(self.root, text="PSIMS v3.7", font=('Arial', 16, 'bold'), bg='gray', fg='white').pack(fill='x', ipady=10)
        
        # --- TABS CONTAINER ---
        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill='both', expand=True, padx=10, pady=10)
        
        self.tab_data = ttk.Frame(self.notebook, padding=10)
        self.tab_analysis = ttk.Frame(self.notebook, padding=10)
        
        self.notebook.add(self.tab_data, text="  1. Data Preparation  ")
        self.notebook.add(self.tab_analysis, text="  2. Analysis & Config  ")
        
        self.build_data_tab()
        self.build_analysis_tab()
        self.build_log_area()
    
    def build_data_tab(self):
        # Section 1: Folders
        lf = ttk.Labelframe(self.tab_data, text="📁 Project Folders", padding=15)
        lf.pack(fill='x', pady=10)
        
        self.create_folder_row(lf, "Raw Input Files:", self.input_folder_var, 'input_folder')
        self.create_folder_row(lf, "CSV Output Folder:", self.csv_folder_var, 'csv_output')
        self.create_folder_row(lf, "Analysis Results:", self.results_folder_var, 'results_output')
        
        # Section 2: Conversion
        lf2 = ttk.Labelframe(self.tab_data, text="🔄 File Conversion Engine", padding=15)
        lf2.pack(fill='x', pady=10)
        
        row1 = ttk.Frame(lf2); row1.pack(fill='x', pady=5)
        ttk.Label(row1, text="PI Batch Files:", width=18).pack(side=LEFT)
        self.lbl_pi = ttk.Label(row1, text="None selected", foreground="gray")
        self.lbl_pi.pack(side=LEFT, padx=10, fill='x', expand=True)
        ttk.Button(row1, text="Select Files", command=self.select_pi_files).pack(side=RIGHT)
        
        row2 = ttk.Frame(lf2); row2.pack(fill='x', pady=5)
        ttk.Label(row2, text="Engagement Files:", width=18).pack(side=LEFT)
        self.lbl_eng = ttk.Label(row2, text="None selected", foreground="gray")
        self.lbl_eng.pack(side=LEFT, padx=10, fill='x', expand=True)
        ttk.Button(row2, text="Select Files", command=self.select_engagement_files).pack(side=RIGHT)
        
        if THEME_AVAILABLE:
            ttk.Button(lf2, text="RUN CONVERSION PIPELINE", command=self.run_conversion, bootstyle="success").pack(pady=15, fill='x')
        else:
            tk.Button(lf2, text="RUN CONVERSION PIPELINE", command=self.run_conversion, bg='green', fg='white').pack(pady=15, fill='x')
            
        self.convert_progress = ttk.Progressbar(lf2, mode='indeterminate', length=300)

    def build_analysis_tab(self):
        lf = ttk.Labelframe(self.tab_analysis, text="⚙️ Configuration", padding=15)
        lf.pack(fill='x', pady=10)
        
        # Grid Layout
        # Row 0: Specialty
        ttk.Label(lf, text="Filter Specialty:", font=('Arial', 10, 'bold'), foreground='#d35400').grid(row=0, column=0, sticky='w', pady=10)
        self.cb_spec = ttk.Combobox(lf, textvariable=self.specialty_var, state='readonly', width=30)
        self.cb_spec.grid(row=0, column=1, padx=10, sticky='w')
        ttk.Button(lf, text="Refresh", command=self.populate_specialties, width=8).grid(row=0, column=2, padx=5, sticky='w')
        
        # Row 1: CSV Selection
        ttk.Button(lf, text="Select Specific CSVs", command=self.select_csv_files).grid(row=1, column=0, pady=10, sticky='w')
        self.lbl_csv = ttk.Label(lf, text="(Default: All CSVs)", foreground="gray")
        self.lbl_csv.grid(row=1, column=1, padx=10, sticky='w')
        
        # Row 2: Modes
        ttk.Label(lf, text="Eligibility Mode:").grid(row=2, column=0, sticky='w', pady=5)
        ttk.Combobox(lf, textvariable=self.eligibility_var, values=['strict','moderate','relaxed','basic'], state='readonly').grid(row=2, column=1, sticky='w', padx=10)
        
        ttk.Checkbutton(lf, text="Require Engagement Data", variable=self.require_eng_var).grid(row=2, column=2, padx=20, sticky='w')
        
        # Row 3: Clusters
        ttk.Label(lf, text="Clusters:").grid(row=3, column=0, sticky='w', pady=5)
        ttk.Spinbox(lf, from_=3, to=10, textvariable=self.clusters_var, width=5).grid(row=3, column=1, sticky='w', padx=10)
        
        ttk.Label(lf, text="Zero Data Pos:").grid(row=3, column=2, sticky='e', pady=5)
        ttk.Combobox(lf, textvariable=self.zero_pos_var, values=['first','last','separate','exclude'], state='readonly', width=10).grid(row=3, column=3, sticky='w', padx=5)
        
        # Row 4: Thresholds
        ttk.Label(lf, text="Thresholds (H/L):").grid(row=4, column=0, sticky='w', pady=5)
        f_thr = ttk.Frame(lf)
        f_thr.grid(row=4, column=1, sticky='w', padx=10)
        ttk.Spinbox(f_thr, from_=20, to=50, textvariable=self.high_thresh_var, width=5).pack(side=LEFT)
        ttk.Spinbox(f_thr, from_=5, to=25, textvariable=self.low_thresh_var, width=5).pack(side=LEFT, padx=5)
        
        # Row 5: Naming
        ttk.Label(lf, text="Naming Method:").grid(row=5, column=0, sticky='w', pady=5)
        ttk.Combobox(lf, textvariable=self.naming_var, values=['absolute','percentile','combined'], state='readonly').grid(row=5, column=1, sticky='w', padx=10)
        
        # Row 6: Toggles
        f_tog = ttk.Frame(lf); f_tog.grid(row=6, column=0, columnspan=4, pady=10, sticky='w')
        ttk.Checkbutton(f_tog, text="Show Metrics", variable=self.show_metrics_var).pack(side=LEFT, padx=5)
        viz_btn = ttk.Checkbutton(f_tog, text="Generate Visualizations", variable=self.generate_viz_var)
        viz_btn.pack(side=LEFT, padx=20)
        if not VISUALIZATION_AVAILABLE: viz_btn.config(state='disabled')

        # Run Button
        if THEME_AVAILABLE:
            self.btn_run = ttk.Button(self.tab_analysis, text="▶ RUN ANALYSIS", command=self.run_analysis, bootstyle="primary", width=25)
        else:
            self.btn_run = tk.Button(self.tab_analysis, text="RUN ANALYSIS", command=self.run_analysis, bg='blue', fg='white', width=25)
        self.btn_run.pack(pady=15)
        
        self.analysis_progress = ttk.Progressbar(self.tab_analysis, mode='indeterminate', length=400)

    def build_log_area(self):
        lf = ttk.Labelframe(self.root, text="System Log", padding=5)
        lf.pack(fill='both', expand=True, padx=10, pady=10)
        self.log_text = ScrolledText(lf, height=10, font=('Consolas', 9))
        self.log_text.pack(fill='both', expand=True)

    def create_folder_row(self, parent, label, var, key):
        f = ttk.Frame(parent); f.pack(fill='x', pady=2)
        ttk.Label(f, text=label, width=18).pack(side=LEFT)
        ttk.Entry(f, textvariable=var).pack(side=LEFT, fill='x', expand=True, padx=5)
        ttk.Button(f, text="...", width=3, command=lambda: self.browse(var, key)).pack(side=LEFT)

    # --- LOGIC METHODS ---
    def log(self, msg):
        print(msg)
        self.log_text.insert(tk.END, str(msg) + "\n")
        self.log_text.see(tk.END)
        self.root.update_idletasks()

    def browse(self, var, key):
        d = filedialog.askdirectory()
        if d: var.set(d); self.config.update_folder(key, d)

    def select_pi_files(self):
        f = filedialog.askopenfilenames()
        if f: self.selected_pi_files = f; self.lbl_pi.config(text=f"{len(f)} files selected", foreground='green')

    def select_engagement_files(self):
        f = filedialog.askopenfilenames()
        if f: self.selected_eng_files = f; self.lbl_eng.config(text=f"{len(f)} files selected", foreground='green')

    def select_csv_files(self):
        d = self.csv_folder_var.get()
        if not os.path.exists(d): return messagebox.showerror("Error", "Set CSV Folder first")
        f = filedialog.askopenfilenames(initialdir=d, filetypes=[("CSV", "*.csv")])
        if f: 
            self.selected_csv_files = [os.path.basename(x) for x in f]
            self.lbl_csv.config(text=f"{len(f)} CSVs selected", foreground='green')

    def run_conversion(self):
        if not self.selected_pi_files and not self.selected_eng_files:
            return messagebox.showwarning("Warning", "No files selected")
        
        self.convert_progress.pack(fill='x', pady=5)
        self.convert_progress.start(10)
        self.log_text.delete(1.0, tk.END)
        
        try:
            conv = SmartConverter(self.csv_folder_var.get(), self.log)
            if self.selected_pi_files: conv.combine_pi_batches(self.selected_pi_files)
            if self.selected_eng_files: conv.convert_engagement_files(self.selected_eng_files)
            self.populate_specialties()
            messagebox.showinfo("Done", "Conversion Complete")
        except Exception as e:
            self.log(f"Error: {e}")
        finally:
            self.convert_progress.stop()
            self.convert_progress.pack_forget()

    def populate_specialties(self):
        p = os.path.join(self.csv_folder_var.get(), 'pi.csv')
        s = ['All Specialties']
        if os.path.exists(p):
            try:
                df = pd.read_csv(p, usecols=lambda c: c.lower() in ['specialty','uin_specialty'])
                # Get first column found
                col = df.columns[0]
                vals = df[col].dropna().unique()
                s += sorted([str(x).strip() for x in vals if str(x).strip()])
                self.log(f"Loaded {len(s)-1} specialties.")
            except: pass
        self.cb_spec['values'] = s
        self.cb_spec.current(0)

    def run_analysis(self):
        self.analysis_progress.pack(fill='x', pady=5)
        self.analysis_progress.start(10)
        
        try:
            # Save configs
            self.config.update_setting('eligibility_mode', self.eligibility_var.get())
            self.config.update_setting('num_clusters', int(self.clusters_var.get()))
            
            engine = PSImsEngine(
                self.csv_folder_var.get(), self.results_folder_var.get(), self.log,
                self.eligibility_var.get(), self.require_eng_var.get(),
                self.naming_var.get(), int(self.high_thresh_var.get()), int(self.low_thresh_var.get()),
                self.show_metrics_var.get(), self.generate_viz_var.get(),
                self.zero_pos_var.get(), self.selected_csv_files, self.specialty_var.get()
            )
            out, prof, smry, viz = engine.run_analysis(int(self.clusters_var.get()))
            
            if out: messagebox.showinfo("Success", "Analysis Complete!")
            else: messagebox.showwarning("Warning", "No output generated")
            
        except Exception as e:
            self.log(f"Error: {e}")
            import traceback
            self.log(traceback.format_exc())
        finally:
            self.analysis_progress.stop()
            self.analysis_progress.pack_forget()


if __name__ == "__main__":
    if THEME_AVAILABLE:
        root = ttk.Window(themename="flatly")
    else:
        root = tk.Tk()
    app = PSImsGUI(root)
    root.mainloop()

KeyboardInterrupt: 